# Tutorial 4: High Density Random Walk
### Learning Objectives
* Packing at high density
* Creating mixed objects
* Sampling chain lengths
* Relaxing structures
### Key Points
* Can be difficult to get initial high denisty packing with full molecules
* Create more random configurations with deterministic approahces
* Testing for stability in simulation engines is trial and error, can verify the packed configuration in mBuild

## Setup
------

In [ ]:
# Import necessary libraries
import mbuild as mb
import numpy as np

from mbuild.path import Path, hard_sphere_random_walk
from mbuild.path.termination import NumSites, Termination, WallTime 
from mbuild.path.constraints import CuboidConstraint

# Check mBuild version
print(f"mBuild version: {mb.__version__}")

mBuild version: 1.3.1


### Hard Sphere Random Walk
-----


In [3]:
# Define system
bond_length = 0.15 # nm
chain_length = 50 # 50 mer
radius = bond_length*0.95 # nm

# Create finish criteria
num_sites = NumSites(chain_length)
walltime = WallTime(20)
conditions = Termination((num_sites, walltime))

# Run a random walk
amorphous_path = hard_sphere_random_walk(
    radius=radius,
    bond_length=bond_length,
    termination=conditions,
    connectivity="linear",
    rw_angles=(0, 3.14),
)

# Visualize
view = amorphous_path.visualize(radius=radius)
view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Pack a Box
-----------

In [17]:
# System Setup
n_chains = 50
num_sites = NumSites(200)
aPath = Path()
volume = CuboidConstraint(7,7,7,pbc=(True, True, True))

# System Runtime
conditions = Termination((num_sites))
aPath = Path()

# setup timer
import time
start = time.perf_counter()
# Fold together
for chain in range(n_chains):
    # Run a random walk
    hard_sphere_random_walk(
        path=aPath,
        radius=0.1,
        bond_length=0.154,
        termination=conditions,
        volume_constraint=volume,
        rw_angles=(3.14/2, 3.14),
    )
print(f"Built {len(aPath.coordinates)} monomers in {(time.perf_counter()-start):.2f}s")

# Visualize
aPath.visualize(hide_periodic_bonds=True)

Built 10000 monomers in 0.95s
Hiding 1560 periodic edges


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Creating a mixture of walks
--------

In [ ]:
# Initialize two random walks
A_name = "_A"
B_name = "_B"

chainDict = {
    A_name: {"bond_length": 0.2, "n_mers":50},
    B_name: {"bond_length": 0.2, "n_mers":200}
}
n_chains = 50
aPath = Path()
volume = CuboidConstraint(8,8,8,pbc=(False, False, False))

# Randomize selection
rng = np.random.default_rng()
chainsList = rng.choice((A_name, B_name), size=n_chains, replace=True)

# setup timer
import time
start = time.perf_counter()
# Fold together
for chain in chainsList:
    # Create finish criteria
    num_sites = NumSites(chainDict[chain]["n_mers"])
    walltime = WallTime(20)
    conditions = Termination((num_sites, walltime))

    # Run a random walk
    hard_sphere_random_walk(
        path=aPath,
        radius=chainDict[chain]["bond_length"]*0.9,
        bond_length=chainDict[chain]["bond_length"],
        termination=conditions,
        volume_constraint=volume,
        rw_angles=(3.14/2, 3.14),
        bead_name=chain,
    )
print(f"Built {len(chainsList)} chains in {(time.perf_counter()-start):.2f}s to build")

# Visualize
aPath.visualize()

Built 50 chains in 0.26s to build


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Build a layered system

In [32]:
# Initialize two random walks
A_name = "_A"
B_name = "_B"

chainDict = {
    A_name: {"bond_length": 0.2, "n_mers":150},
    B_name: {"bond_length": 0.2, "n_mers":200}
}
aPath = Path()
chainsList = [A_name] * 12 + [B_name] * 8 +[A_name] * 12 + [B_name] * 8 +[A_name] * 12


# Randomize selection
rng = np.random.default_rng()

# setup timer
import time
start = time.perf_counter()
# Fold together
slab = 0
prev_chain = "_B"
for i,chain in enumerate(chainsList):
    if prev_chain != chain:
        volume = CuboidConstraint(8,8,1.5, center=(0,0,slab), pbc=(False, False, False))
        slab += 1.5
        prev_chain = chain
    # Create finish criteria
    num_sites = NumSites(chainDict[chain]["n_mers"])
    walltime = WallTime(20)
    conditions = Termination((num_sites, walltime))

    # Run a random walk
    hard_sphere_random_walk(
        path=aPath,
        radius=chainDict[chain]["bond_length"]*0.5,
        bond_length=chainDict[chain]["bond_length"],
        termination=conditions,
        volume_constraint=volume,
        rw_angles=(3.14/2, 3.14),
        bead_name=chain,
        tolerance=0.01
    )
print(f"Built {len(chainsList)} chains in {(time.perf_counter()-start):.2f}s to build")

# Visualize
aPath.visualize()

Built 52 chains in 0.71s to build


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Termination Critera
-----

In [1]:
# Initialize random walk

# Sample different chain lengths

# Set wall-time criteria

## Relaxing final structures
-----

In [8]:
# Create an unstable box

# Print out overlaps

# Relax using simulation.py

### Exercise: 